In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'orjson', 'polars', 'pyarrow'])
import os, gc, gzip, orjson, random, csv
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('config.py')))
from config import *
from gdrive_stream import stream_jsonl_filtered, preview_jsonl
print_config()

# ── CÁC PATH ĐÃ ĐƯỢC ĐỊNH NGHĨA TRONG config.py ──────────────
# REVIEW_FILE_ID, META_FILE_ID  → stream từ Google Drive
# TRAIN_PATH, TEST_PATH, META_PATH, MAPPING_PATH → lưu local


In [4]:
def extract_valid_fashion_items():
    """Quét Meta JSONL trên Drive để lấy ASIN thuộc AMAZON FASHION."""
    print("BƯỚC 0: Quét Meta Data từ Google Drive (streaming)...")
    valid_fashion_asins = set()
    for data in stream_jsonl_filtered(
        META_FILE_ID,
        filter_fn=lambda d: d.get('main_category') == 'AMAZON FASHION',
        fields=['parent_asin']
    ):
        asin = data.get('parent_asin')
        if asin:
            valid_fashion_asins.add(asin)
    print(f"-> Tìm thấy {len(valid_fashion_asins):,} sản phẩm AMAZON FASHION.")
    return valid_fashion_asins


In [5]:
def jsonl_to_parquet_filtered(output_path, chunk_size, valid_fashion_asins):
    """Stream Review JSONL từ Drive, lọc Fashion, ghi ra Parquet."""
    print("BƯỚC 1: Stream Review từ Google Drive → Parquet (chỉ Fashion)...")
    writer = None
    buffer = []
    chunk_count = 0

    for data in stream_jsonl_filtered(
        REVIEW_FILE_ID,
        filter_fn=lambda d: d.get('parent_asin') in valid_fashion_asins,
        fields=['user_id', 'parent_asin', 'rating', 'timestamp', 'helpful_vote', 'verified_purchase']
    ):
        buffer.append({
            'user_id':           data.get('user_id'),
            'parent_asin':       data.get('parent_asin'),
            'rating':            data.get('rating'),
            'timestamp':         data.get('timestamp'),
            'helpful_vote':      data.get('helpful_vote', 0),
            'verified_purchase': data.get('verified_purchase', False),
        })

        if len(buffer) >= chunk_size:
            chunk_count += 1
            table = pa.Table.from_pandas(pd.DataFrame(buffer))
            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema)
            writer.write_table(table)
            buffer.clear()
            print(f"   Đã ghi Chunk {chunk_count}")
            gc.collect()

    if buffer:
        table = pa.Table.from_pandas(pd.DataFrame(buffer))
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)
        writer.write_table(table)

    if writer:
        writer.close()
    print("-> Hoàn tất stream Review!")


In [6]:
def apply_k_core(data_path, k=5):
    print(f"BƯỚC 2: Áp dụng K-Core={k} (Tự động lặp cho đến khi hội tụ)...")
    lf = pl.scan_parquet(data_path).select(['user_id', 'parent_asin'])
    curr_data = lf.collect()
    
    iteration = 1
    while True:
        print(f"   Vòng lặp K-Core thứ {iteration}...")
        
        # Ghi nhận số lượng dòng hiện tại trước khi lọc
        prev_height = curr_data.height
        
        # 1. Lọc User có ít nhất K tương tác
        user_counts = curr_data.group_by('user_id').len()
        valid_users = user_counts.filter(pl.col('len') >= k).select('user_id')
        curr_data = curr_data.join(valid_users, on='user_id', how='inner')

        # 2. Lọc Item có ít nhất K tương tác
        item_counts = curr_data.group_by('parent_asin').len()
        valid_items = item_counts.filter(pl.col('len') >= k).select('parent_asin')
        curr_data = curr_data.join(valid_items, on='parent_asin', how='inner')

        del user_counts, valid_users, item_counts, valid_items
        gc.collect()
        
        # Lấy các dòng duy nhất
        curr_data = curr_data.unique()
        
        # Lấy số lượng dòng sau khi lọc
        curr_height = curr_data.height
        
        print(f"      -> Số tương tác còn lại: {curr_height:,} (Đã xóa {prev_height - curr_height:,} dòng)")
        
        # ĐIỀU KIỆN HỘI TỤ: Nếu số dòng không giảm đi nữa thì dừng lại
        if curr_height == prev_height:
            print(f"   => K-Core ĐÃ HỘI TỤ SAU {iteration} VÒNG LẶP!")
            break
            
        iteration += 1

    return curr_data

In [7]:
def split_and_map(raw_path, valid_ids, train_out, test_out, ratio):
    print("BƯỚC 3: Map ID và Phân tách Train/Test theo MỐC THỜI GIAN CHUNG...")

    valid_user_set = set(valid_ids['user_id'].to_list())
    valid_item_set = set(valid_ids['parent_asin'].to_list())

    u_map = pl.DataFrame({'user_id': list(valid_user_set)}).with_row_index('mapped_user_id', offset=1)
    i_map = pl.DataFrame({'parent_asin': list(valid_item_set)}).with_row_index('mapped_item_id', offset=1)
    
    u_map = u_map.with_columns(pl.col('mapped_user_id').cast(pl.Int32))
    i_map = i_map.with_columns(pl.col('mapped_item_id').cast(pl.Int32))

    # Nạp dữ liệu và gắn ID. Bây giờ lấy thêm cả 2 cột mới
    df_clean = (pl.scan_parquet(raw_path)
                .join(u_map.lazy(), on='user_id', how='inner')
                .join(i_map.lazy(), on='parent_asin', how='inner')
                .select(['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp', 'helpful_vote', 'verified_purchase'])
                .collect())

    df_clean = df_clean.sort('timestamp')
    
    cutoff_idx = int(df_clean.height * ratio)
    
    cutoff_timestamp = df_clean['timestamp'][cutoff_idx]
    print(f"-> Mốc thời gian chung để cắt hệ thống là: {cutoff_timestamp}")

    train_data = df_clean.filter(pl.col('timestamp') <= cutoff_timestamp)
    test_data  = df_clean.filter(pl.col('timestamp') > cutoff_timestamp)

    print("-> Đang loại bỏ các User/Item mới xuất hiện trong Test mà Train chưa có...")
    train_users = train_data.select('mapped_user_id').unique()
    train_items = train_data.select('mapped_item_id').unique()
    
    test_data = (test_data
                 .join(train_users, on='mapped_user_id', how='inner')
                 .join(train_items, on='mapped_item_id', how='inner'))

    # Ghi xuống đĩa cứng
    train_data.write_parquet(train_out)
    test_data.write_parquet(test_out)

    print(f"-> Train: {train_data.height:,} dòng | Test: {test_data.height:,} dòng.")
    return valid_item_set, i_map

In [8]:
def process_meta(valid_items_set, item_map_df, output_path, chunk_size):
    import re
    print("BƯỚC 4: Stream và Xử lý Meta Data từ Google Drive...")
    writer = None
    buffer = []
    
    def clean_number(val):
        try:
            if isinstance(val, str):
                nums = re.findall(r'\d+\.?\d*', val)
                return float(nums[0]) if nums else 0.0
            return float(val) if val is not None else 0.0
        except:
            return 0.0

    with open(meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                data = orjson.loads(line)
                asin = data.get('parent_asin')

                if asin in valid_items_set:
                    buffer.append({
                        'parent_asin': asin,
                        'price': clean_number(data.get('price')),
                        'average_rating': clean_number(data.get('average_rating')),
                        'rating_number': int(clean_number(data.get('rating_number'))),
                        'store': str(data.get('store', 'Unknown')),
                        'categories': str(data.get('categories', '[]'))
                    })

                if len(buffer) >= chunk_size:
                    df_chunk = pd.DataFrame(buffer)
                    table = pa.Table.from_pandas(df_chunk)
                    if writer is None:
                        writer = pq.ParquetWriter(output_path + ".tmp", table.schema)
                    writer.write_table(table)
                    buffer.clear()
                    gc.collect()

            except Exception:
                continue

    if buffer:
        df_chunk = pd.DataFrame(buffer)
        table = pa.Table.from_pandas(df_chunk)
        if writer is None:
            writer = pq.ParquetWriter(output_path + ".tmp", table.schema)
        writer.write_table(table)

    if writer: writer.close()

    print("   Đang gắn Map ID cho Meta Data...")
    (pl.scan_parquet(output_path + ".tmp")
     .join(item_map_df.lazy(), on='parent_asin', how='inner')
     .drop('parent_asin') 
     .sink_parquet(output_path))

    os.remove(output_path + ".tmp")
    print("-> Hoàn tất xử lý Meta Data!")

In [9]:
valid_fashion_asins = extract_valid_fashion_items()
jsonl_to_parquet_filtered(INTERIM_RAW_PATH, CHUNK_SIZE, valid_fashion_asins)
valid_ids_df = apply_k_core(INTERIM_RAW_PATH, K_CORE)
valid_items, item_mapping = split_and_map(INTERIM_RAW_PATH, valid_ids_df, TRAIN_PATH, TEST_PATH, TRAIN_RATIO)
item_mapping.write_parquet(MAPPING_PATH)
print(f"-> Đã lưu Mapping tại: {MAPPING_PATH}")
process_meta(valid_items, item_mapping, META_PATH, CHUNK_SIZE)

# Dọn dẹp file tạm
if os.path.exists(INTERIM_RAW_PATH):
    os.remove(INTERIM_RAW_PATH)
    print("-> Đã xóa file tạm.")


BƯỚC 0: Quét Meta Data từ Google Drive (streaming)...


NameError: name 'stream_jsonl_filtered' is not defined